## Prepare Land Registry Data

In [1]:
import duckdb
from pathlib import Path
from data_importers import LandRegistryDataImporter, OnsPostcodeDirectoryImporter

## Download Data

In [ ]:
LAND_REGISTRY_DATA_PATH = Path("../../data/land_registry/")
assert LAND_REGISTRY_DATA_PATH.parent.exists(), f"Data path {LAND_REGISTRY_DATA_PATH} does not exist. Please update the path to point to the correct location of the land registry data."

In [ ]:
land_registry_data = LandRegistryDataImporter(LAND_REGISTRY_DATA_PATH)

In [ ]:
land_registry_data.download_land_registry_data(2023, 2026)

In [ ]:
ONS_PD_DATA_PATH = Path("../../data/ons_postcode_directory/")

In [ ]:
ons_postcode_directory_data = OnsPostcodeDirectoryImporter(ONS_PD_DATA_PATH)

## Read Data

In [ ]:
land_registry_data = duckdb.sql(
    f"""
    SELECT
        column00 AS id,
        column01 AS price,
        column02 AS date,
        column03 AS postcode,
        column04 AS property_type,
        column05 AS old_new,
        column06 AS duration,
        column07 AS paon,
        column08 AS saon,
        column09 AS street,
        column10 AS locality,
        column11 AS town_city,
        column12 AS district,
        column13 AS county,
        column14 AS ppd_category,
        column15 AS record_type
    FROM read_csv('{LAND_REGISTRY_DATA_PATH}/pp-*.csv', header = false)
    """
)

In [ ]:
land_registry_data

In [ ]:
duckdb.sql("DESCRIBE FROM land_registry_data")

In [ ]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT * REPLACE (
        CASE property_type
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE property_type
        END AS property_type
    )
    FROM land_registry_data
    """
)

In [ ]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT * FROM land_registry_data
    WHERE property_type != 'Other'
    """
)

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT *, year(date) AS year
    FROM land_registry_data
    """
)

In [ ]:
duckdb.sql(
    """
    SELECT id, date, year, property_type, price
    FROM land_registry_data
    """
)

In [ ]:
annual_price_by_property_type = duckdb.sql(
    """
    SELECT
        year,
        property_type,
        MEDIAN(price) AS median_price
    FROM land_registry_data
    GROUP BY year, property_type
    ORDER BY year, property_type
    """
)

In [ ]:
annual_price_by_property_type

## Visualise the results

In [ ]:
annual_price_by_property_type.pl()

In [ ]:
import plotly.express as px

In [ ]:
px.line(
    annual_price_by_property_type.pl(),
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [ ]:
from pathlib import Path
Path("../../data/price_paid_insights").mkdir(parents=True, exist_ok=True)

duckdb.sql(
    """
    COPY annual_price_by_property_type
    TO '../../data/price_paid_insights/annual_price_by_property_type.parquet'
    (FORMAT PARQUET)
    """
)

In [ ]:
duckdb.sql(
    f"""
    SELECT
        year(column02) AS year,
        CASE column04
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE column04
        END AS property_type,
        MEDIAN(column01) AS median_price
    FROM read_csv('{LAND_REGISTRY_DATA_PATH}/pp-*.csv', header = false)
    WHERE column04 != 'O'
    GROUP BY ALL
    ORDER BY year, property_type
    """
)

In [ ]:
full_query = f"""
    SELECT
        year(column02) AS year,
        CASE column04
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE column04
        END AS property_type,
        MEDIAN(column01) AS median_price
    FROM read_csv('{LAND_REGISTRY_DATA_PATH}/pp-*.csv', header = false)
    WHERE column04 != 'O'
    GROUP BY ALL
    ORDER BY year, property_type
"""

In [ ]:
duckdb.sql(f"EXPLAIN {full_query}")

In [ ]:
duckdb.sql(full_query)